In [1]:
import xarray as xr
import numpy as np
import dask
from dask.distributed import Client
from dask_jobqueue import PBSCluster

# PBS

In [2]:
cluster = PBSCluster(
    cores=1,
    memory='32GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='1:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.176:38307,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# defs

In [3]:
def calc_RH(T, Td):
    # Using August-Roche-Magnus approximation for sat. vapor pressure
    # calculate RH from T and Td in C
    a = 17.625
    b = 243.04
    vp_a = xr.ufuncs.exp(((a * Td) / (b + Td)))
    vp_s = xr.ufuncs.exp(((a * T)/(b + T)))
    rh = ((vp_a / vp_s) * 100)
    rh.rename('relative_humidity')
    return rh

# data import

In [4]:
hi_ds = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/hourly_HI')
hi_ds

<xarray.Dataset> Size: 19GB
Dimensions:    (time: 755304, latitude: 61, longitude: 101)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 488B 10.0 10.25 10.5 10.75 ... 24.5 24.75 25.0
  * longitude  (longitude) float64 808B -85.0 -84.75 -84.5 ... -60.25 -60.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 19GB dask.array<chunksize=(22888, 61, 101), meta=np.ndarray>

In [5]:
td_ds = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_dew')
td_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2D     (time, latitude, longitude) float32 30GB dask.array<chunksize=(22888, 82, 121), meta=np.ndarray>

In [6]:
t_ds = xr.open_zarr('/glade/work/acruz/Caribbean_Heat_data/ERA5/sfc_hourly_temp')
t_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (time, latitude, longitude) float32 30GB dask.array<chunksize=(22888, 82, 121), meta=np.ndarray>

# calcs

In [7]:
rh_ds = calc_RH(t_ds['VAR_2T'], td_ds['VAR_2D'])
rh_ds

<xarray.DataArray (time: 755304, latitude: 82, longitude: 121)> Size: 30GB
dask.array<mul, shape=(755304, 82, 121), dtype=float32, chunksize=(22888, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

# normal monthly clim

In [10]:
me_rh = rh_ds.groupby('time.month').mean()
me_rh.compute()
me_rh

<xarray.DataArray (month: 12, latitude: 82, longitude: 121)> Size: 476kB
dask.array<stack, shape=(12, 82, 121), dtype=float32, chunksize=(1, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [9]:
me_t = t_ds.groupby('time.month').mean()
me_t.compute()
me_t

<xarray.Dataset> Size: 478kB
Dimensions:    (month: 12, latitude: 82, longitude: 121)
Coordinates:
  * month      (month) int64 96B 1 2 3 4 5 6 7 8 9 10 11 12
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    VAR_2T     (month, latitude, longitude) float32 476kB dask.array<chunksize=(1, 82, 121), meta=np.ndarray>

## export

In [11]:
me_rh.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/ERA5_clim_RH.nc')

In [13]:
me_t.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/ERA5_clim_T.nc')

In [16]:
rh_ds.to_netcdf('/glade/work/acruz/Caribbean_Heat_data/ERA5/ERA5_RH.nc')

# quit

In [14]:
client.shutdown()